Данные берем из соревнования H&M: https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data?select=images

In [9]:
import pandas as pd
from tqdm.auto import tqdm

sample = pd.read_csv('transactions_train.csv', nrows=100_000)
sample['t_dat'] = pd.to_datetime(sample['t_dat'])


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import torch
from torch.utils.data import DataLoader, Dataset

class HMDataset(Dataset):
    def __init__(self, data, max_seq_len=50):
        data = data.sort_values(by=['customer_id', 't_dat'])
        users_interactions = data['customer_id'].value_counts()
        self.good_users = users_interactions[users_interactions >= 5].index
        data = data[data['customer_id'].isin(self.good_users)]
        self.all_items = data['article_id'].unique()

        self.customer_map = {cid: idx+1 for idx, cid in enumerate(data['customer_id'].unique())}
        self.item_map = {iid: idx+1 for idx, iid in enumerate(data['article_id'].unique())}

        data['customer_id'] = data['customer_id'].map(self.customer_map)
        data['article_id'] = data['article_id'].map(self.item_map)

        self.sequences = data.groupby('customer_id')['article_id'].apply(list).tolist()
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq = seq[-self.max_seq_len-1:]

        input_seq = [0] * self.max_seq_len
        target_seq = [0] * self.max_seq_len
        if len(seq) >= 2:
            input_tokens = seq[:-1][-self.max_seq_len:]
            target_tokens = seq[1:][-self.max_seq_len:]
            input_seq[-len(input_tokens):] = input_tokens
            target_seq[-len(target_tokens):] = target_tokens

        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)


In [12]:
dataset = HMDataset(sample)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

In [13]:
import torch.optim as optim
import pandas as pd

class SASRec(nn.Module):
    def __init__(self, item_num, hidden_dim=64, max_seq_len=50, num_blocks=2, num_heads=2, dropout_rate=0.2):
        super(SASRec, self).__init__()
        self.item_embedding = nn.Embedding(item_num + 1, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, dropout=dropout_rate)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)

        self.output_layer = nn.Linear(hidden_dim, item_num + 1)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_seq):
        positions = torch.arange(input_seq.size(1), device=input_seq.device).unsqueeze(0).expand_as(input_seq)
        seq_emb = self.item_embedding(input_seq) + self.position_embedding(positions)
        seq_emb = self.dropout(seq_emb).transpose(0, 1)  # [seq_len, batch_size, hidden_dim]

        transformer_output = self.transformer(seq_emb)
        transformer_output = transformer_output.transpose(0, 1)  # [batch_size, seq_len, hidden_dim]

        logits = self.output_layer(transformer_output)
        return logits

In [41]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
max_seq_len = 50
item_num = max([max(seq) for seq in dataset.sequences if seq])
model = SASRec(item_num=item_num, max_seq_len=max_seq_len)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [42]:
epochs = 10
lr = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for input_seq, target_seq in tqdm(dataloader, leave=False):
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)

        optimizer.zero_grad()
        logits = model(input_seq)
        logits = logits.view(-1, item_num + 1)
        targets = target_seq.view(-1)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 9.2039


Epoch 2/10, Loss: 8.8128


Epoch 3/10, Loss: 8.7902


Epoch 4/10, Loss: 8.7803


Epoch 5/10, Loss: 8.7765


Epoch 6/10, Loss: 8.7688


Epoch 7/10, Loss: 8.7671


Epoch 8/10, Loss: 8.7636


Epoch 9/10, Loss: 8.7563


Epoch 10/10, Loss: 8.7557


In [19]:
import torch.nn as nn
from torchvision import transforms, models
import torch 
import os
import random
import numpy as np
from PIL import Image



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet18(pretrained=True)
model.fc = nn.Identity()
model = model.to(device)
model.eval()
image_folder = "images"
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

embedding_dim = 512
embeddings = np.zeros((len(dataset.all_items) + 1, embedding_dim), dtype=np.float32)
with torch.no_grad():
    for item_id in tqdm(dataset.all_items):
        subfolder = '0'+str(item_id)[:2]  # например, '089'
        img_path = os.path.join(image_folder, subfolder, f"0{item_id}.jpg")
        if not os.path.exists(img_path):
            continue
        try:
            image = Image.open(img_path).convert('RGB')
            image_tensor = preprocess(image).unsqueeze(0).to(device)
            embedding = model(image_tensor).squeeze(0).cpu().numpy()
            embeddings[dataset.item_map[item_id]] = embedding
        except Exception as e:
            print(f"Ошибка при обработке {img_path}: {e}")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 12635/12635 [06:10<00:00, 34.08it/s]


In [35]:
class ImageSASRec(nn.Module):
    def __init__(self, 
                 item_num, 
                 image_embeddings : list[np.ndarray],
                 fusion_type='concat',
                 freeze_embeds=False,
                 hidden_dim=64,
                 max_seq_len=50,
                 num_blocks=2,
                 num_heads=2,
                 dropout_rate=0.2):
        super().__init__()
        self.item_embedding = nn.Embedding(item_num + 1, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        
        # тут начинаются отличия
        self.image_embeddings = nn.Embedding.from_pretrained(image_embeddings, freeze=freeze_embeds)
        self.image_project = nn.Linear(embedding_dim, hidden_dim)
        self.fusion_layer = nn.Linear(hidden_dim , hidden_dim)
        if fusion_type in ['concat', 'add']:
            self.fusion_type = fusion_type
            if fusion_type == 'concat':
                self.fusion_layer = nn.Linear(2*hidden_dim , hidden_dim)
        # тут заканчиваются отличия
                
                
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, dropout=dropout_rate)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)

        self.output_layer = nn.Linear(hidden_dim, item_num + 1)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_seq):
        positions = torch.arange(input_seq.size(1), device=input_seq.device).unsqueeze(0).expand_as(input_seq)
        seq_emb = self.item_embedding(input_seq) + self.position_embedding(positions) 
        
        # тут начинаются отличия        
        image_embeds = self.image_project(self.image_embeddings(input_seq))
        if self.fusion_type == 'concat':
            seq_emb = torch.cat((seq_emb,image_embeds), dim=-1)
        elif self.fusion_type == 'add':
            seq_emb = seq_emb + image_embeds
        seq_emb = self.fusion_layer(seq_emb)
        # тут заканчиваются отличия
    
        seq_emb = self.dropout(seq_emb).transpose(0, 1)  # [seq_len, batch_size, hidden_dim]

        transformer_output = self.transformer(seq_emb)
        transformer_output = transformer_output.transpose(0, 1)  # [batch_size, seq_len, hidden_dim]

        logits = self.output_layer(transformer_output)
        return logits

In [36]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
max_seq_len = 50
model = ImageSASRec(item_num=item_num, image_embeddings=torch.from_numpy(embeddings), freeze_embeds=True, fusion_type='add', max_seq_len=max_seq_len)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [37]:
epochs = 10
lr = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for input_seq, target_seq in tqdm(dataloader, leave=False):
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)

        optimizer.zero_grad()
        logits = model(input_seq)
        logits = logits.view(-1, item_num + 1)
        targets = target_seq.view(-1)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 9.2000


Epoch 2/10, Loss: 8.8084


Epoch 3/10, Loss: 8.7860


Epoch 4/10, Loss: 8.7796


Epoch 5/10, Loss: 8.7730


Epoch 6/10, Loss: 8.7675


Epoch 7/10, Loss: 8.7637


Epoch 8/10, Loss: 8.7611


Epoch 9/10, Loss: 8.7595


Epoch 10/10, Loss: 8.7569


In [38]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
max_seq_len = 50
model = ImageSASRec(item_num=item_num, image_embeddings=torch.from_numpy(embeddings), freeze_embeds=True, fusion_type='concat', max_seq_len=max_seq_len)

In [39]:
epochs = 10
lr = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for input_seq, target_seq in tqdm(dataloader, leave=False):
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)

        optimizer.zero_grad()
        logits = model(input_seq)
        logits = logits.view(-1, item_num + 1)
        targets = target_seq.view(-1)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 9.2058


Epoch 2/10, Loss: 8.8117


Epoch 3/10, Loss: 8.7885


Epoch 4/10, Loss: 8.7795


Epoch 5/10, Loss: 8.7720


Epoch 6/10, Loss: 8.7713


Epoch 7/10, Loss: 8.7612


Epoch 8/10, Loss: 8.7624


Epoch 9/10, Loss: 8.7576


Epoch 10/10, Loss: 8.7549
